# Deriving the Gillespie Algorithm from Scratch

**Goal of this notebook.** Gillespie's Stochastic Simulation Algorithm (SSA) generates
exact random trajectories of a chemically reacting system. At its heart it uses just
**two random numbers** per step:

* $r_1$ decides **when** the next reaction happens,
* $r_2$ decides **which** reaction happens.

This notebook **derives** both of those rules from a single physical idea (the *propensity*),
explaining every step. If you have never seen the algorithm before, you are the intended reader.
You need only basic probability (the notions of a probability density and a cumulative distribution)
and a little calculus.

> **Why bother with randomness at all?** In a living cell some molecules exist in only a few
> copies. Reactions then fire as *discrete, random* events, and two genetically identical cells
> can diverge purely by chance. The smooth deterministic rate equations (ODEs) you may know
> describe only the *average* behaviour and break down at low copy numbers. The SSA captures
> the actual stochastic dynamics. (As a consistency check: averaged over many SSA runs, the
> trajectories converge to those same ODEs when molecule counts are large.)

## 1. The physical picture

Imagine $N$ kinds of molecules bouncing around in a well-stirred container of volume $V$.
We assume the mixture is **well-stirred**, meaning molecules are scattered uniformly so that
*where* they are does not matter — only *how many* of each there are. The state of the system
is therefore the **molecule-count vector**

$$\mathbf{X}(t) = (X_1(t),\, X_2(t),\, \dots,\, X_N(t)).$$

A reaction fires whenever the right molecules collide with enough energy. Because thermal
motion is random, **when** a given reaction fires is a random variable. Crucially, the process
is **memoryless** (Markovian): what happens next depends only on the current counts $\mathbf{X}(t)$,
not on the past. Given the current state, we only need to answer two questions to advance the
simulation one step:

1. How long until the next reaction?  (a waiting time $\tau$)
2. Which reaction is it?  (an index $j$)

## 2. The propensity $a_j$ — the one assumption everything rests on

For each reaction channel $R_j$ ($j = 1,\dots,M$) we define a number $a_j(\mathbf{x})$, the
**propensity**:

$$\boxed{\;a_j(\mathbf{x})\,dt \;=\; \Pr\!\left\{\,R_j \text{ fires once in the next } [t,t+dt) \;\middle|\; \mathbf{X}(t)=\mathbf{x}\,\right\}\;}$$

Read this as: *"$a_j$ times a tiny time $dt$ is the probability that reaction $j$ goes off in
the next instant."* Here $dt$ is infinitesimal, so $a_j\,dt$ is small and we ignore the
negligible chance of two reactions firing in the same $dt$.

**Intuition.** $a_j(\mathbf{x})$ measures *how eager* reaction $j$ is right now. More reactant
molecules $\Rightarrow$ more possible collisions per second $\Rightarrow$ a larger $a_j$. The
propensity is the stochastic analogue of a deterministic rate: it is a *probability per unit time*.

We also define the **total propensity**

$$a_0(\mathbf{x}) \;=\; \sum_{j=1}^{M} a_j(\mathbf{x}),$$

so that $a_0\,dt$ is the probability that *some* reaction (any of them) fires in the next $dt$.

### Where the propensity formulas come from

For *elementary* reactions, $a_j$ is just **(number of distinct reacting pairs) × (per-pair reaction rate)**.
Counting pairs gives the familiar forms:

**Unimolecular** $A \to \cdots$. Each of the $X_A$ molecules of $A$ decays independently, so there are
$X_A$ independent "opportunities":

$$a = c\,X_A.$$

**Hetero-bimolecular** $A + B \to \cdots$. An $A$ can pair with any of the $X_B$ molecules of $B$,
and there are $X_A$ choices of $A$, giving $X_A X_B$ distinct $A$–$B$ pairs:

$$a = c\,X_A X_B.$$

**Homo-bimolecular** $2A \to \cdots$. We must choose *two distinct* $A$ molecules, and order does
not matter. The count is $\dbinom{X_A}{2} = \tfrac{1}{2}X_A(X_A-1)$ — you cannot pair a molecule
with itself, and the pair $(A_i, A_j)$ is the same as $(A_j, A_i)$:

$$a = c\,\frac{X_A(X_A-1)}{2}.$$

> *Example.* With $X_A = 3$ molecules of $A$ and $X_B = 2$ of $B$, there are $3\times 2 = 6$
> possible $A$–$B$ collisions, so $a = 6c$. The propensity is literally "how many pairs could react,
> times how fast each pair reacts." The constant $c$ packages up the physics (temperature, volume,
> molecular sizes).

## 3. The object we need to find: the joint density $p(\tau, j)$

Because the process is memoryless, the *entire* next step is captured by one joint probability
density: let $p(\tau, j)\,d\tau$ be the probability that

* the **next** reaction is $R_j$, **and**
* it fires in the narrow window $[t+\tau,\; t+\tau+d\tau)$.

If we can write $p(\tau, j)$ in a form we recognise, we can sample $(\tau, j)$ from it — and that
*is* one step of the algorithm. The rest of the derivation is just computing $p(\tau, j)$.

We do it in two halves: first the timing (get $\tau$), then the identity (get $j$).

## 4. Step 1 — the probability that *nothing* happens for a while

The event "the next reaction occurs around time $\tau$" can be split into two pieces:

* **nothing** fires during the whole interval $[t,\, t+\tau)$, and then
* **something** fires in the tiny window $[t+\tau,\, t+\tau+d\tau)$.

So we first need $P_0(\tau)$, the probability that **no** reaction at all fires during $[t, t+\tau)$.

**The key observation.** If nothing has fired yet, the state has not changed, so every propensity
is still $a_j(\mathbf{x})$, and the probability that *some* reaction fires in an interval of length
$\Delta\tau$ is $a_0\,\Delta\tau$. Hence the probability that **none** fires is $1 - a_0\,\Delta\tau$.

**Discrete argument (the intuitive route).** Chop $[0,\tau]$ into $n$ slices each of width
$\tau/n$. Assuming independence across slices (justified by the memoryless property), the
probability of surviving all $n$ slices is

$$P_0(\tau) \;\approx\; \left(1 - a_0\,\frac{\tau}{n}\right)^{\!n}.$$

Now let $n\to\infty$. This is the classic limit that defines the exponential,
$\displaystyle\lim_{n\to\infty}(1 + x/n)^n = e^{x}$, with $x = -a_0\tau$:

$$\boxed{\;P_0(\tau) \;=\; e^{-a_0 \tau}\;}$$

*(Equivalent ODE route: $P_0(\tau+d\tau) = P_0(\tau)(1-a_0\,d\tau)$ gives
$dP_0/d\tau = -a_0\,P_0$ with $P_0(0)=1$, whose solution is $e^{-a_0\tau}$.)*

**What this means.** $P_0(\tau)$ is a decaying curve: the longer you wait, the less plausible it
is that still nothing has happened. With $a_0 = 0$ it stays $1$ forever (no reactions possible);
with large $a_0$ it crashes to $0$ quickly (busy system).

## 5. Step 2 — the waiting time $\tau$ is Exponential$(a_0)$

Now combine the two pieces above. The probability that the next reaction lands in
$[t+\tau,\, t+\tau+d\tau)$ is

$$\underbrace{P_0(\tau)}_{\text{nothing for }\tau}\;\times\;\underbrace{a_0\,d\tau}_{\text{something in }d\tau}
\;=\; e^{-a_0\tau}\,a_0\,d\tau.$$

So the **probability density** of the waiting time is

$$\boxed{\;f(\tau) \;=\; a_0\,e^{-a_0\tau}, \qquad \tau \ge 0\;}$$

which is precisely the **exponential distribution with rate $a_0$**. We write $\tau \sim \mathrm{Exp}(a_0)$.
Its mean and variance are

$$\mathbb{E}[\tau] = \frac{1}{a_0}, \qquad \mathrm{Var}(\tau) = \frac{1}{a_0^2}.$$

**Reading this physically.** The waiting time is usually short (the density is largest near $\tau=0$)
but has a long tail (occasionally you wait a long time). Doubling the total reaction rate $a_0$
halves the expected wait. This is exactly the *memoryless* property: no matter how long you have
already waited, the *remaining* wait has the same distribution. That sounds paradoxical but is
correct here — the molecules have no memory, so the collision rate depends only on the current
counts, not on how long they have been trying.

## 6. How $\tau$ is actually obtained — and what $r_1$ is

We have shown $\tau$ is exponential, but how do we *draw* a random $\tau$ on a computer?
The standard trick is **inverse-CDF sampling**, which works for any distribution:

1. Compute the **cumulative distribution function** (CDF), the probability that the waiting
   time is at most $\tau$:
   $$F(\tau) \;=\; \Pr(T \le \tau) \;=\; \int_0^\tau a_0\,e^{-a_0 s}\,ds \;=\; 1 - e^{-a_0\tau}.$$
   Notice $F$ rises smoothly from $0$ (at $\tau=0$) to $1$ (as $\tau\to\infty$).
2. Draw a uniform random number $r_1 \sim U(0,1)$ and set $F(\tau) = r_1$, then **invert** to
   read off $\tau$.

**Why inverting works.** If $r_1$ is spread evenly on $(0,1)$ and we set $\tau = F^{-1}(r_1)$,
then the fraction of samples landing below any value $\tau$ equals $F(\tau)$ — so $\tau$ ends up
distributed exactly according to $F$. Intuitively, regions where $F$ is steep (high density)
"soak up" more of the uniform interval and so get more samples.

Solving $1 - e^{-a_0\tau} = r_1$ for $\tau$:

$$e^{-a_0\tau} = 1 - r_1 \;\;\Longrightarrow\;\; \tau = -\frac{1}{a_0}\ln(1 - r_1).$$

Because $1 - r_1$ is *also* uniform on $(0,1)$, this is conventionally written

$$\boxed{\;\tau \;=\; \frac{1}{a_0}\,\ln\!\left(\frac{1}{r_1}\right)\;}, \qquad r_1 \sim U(0,1).$$

> **So what is $r_1$?** A single uniform random number on $(0,1)$ — the *only* source of randomness
> for the **timing**. Pushed through the formula above, a flat die-roll becomes an
> exponentially-distributed waiting time. The new simulation clock is then advanced to
> $t_{\text{new}} = t + \tau$.

## 7. Step 3 — which reaction fires: $p(\tau,j)$ factors beautifully

We now fold in the reaction identity. Suppose a reaction *does* fire in the window
$[t+\tau, t+\tau+d\tau)$. The probability that it is *specifically* $R_j$ (as opposed to one of the
others) is its propensity divided by the total:

$$\Pr(\text{it is } R_j \mid \text{something fires}) \;=\; \frac{a_j\,d\tau}{a_0\,d\tau} \;=\; \frac{a_j}{a_0}.$$

Multiplying the "survive then fire" probability by this conditional gives the joint density:

$$p(\tau, j) \;=\; \underbrace{e^{-a_0\tau}}_{P_0(\tau)}\;\cdot\; a_j \;=\; a_j\,e^{-a_0\tau}.$$

**The crucial algebraic move — factor out $a_0$:**

$$p(\tau, j) \;=\; \underbrace{\bigl(a_0\,e^{-a_0\tau}\bigr)}_{\text{only involves }\tau}\;\cdot\;\underbrace{\bigl(\tfrac{a_j}{a_0}\bigr)}_{\text{only involves }j}.$$

A joint density that splits into a piece depending only on $\tau$ times a piece depending only on
$j$ means the two variables are **statistically independent**. This has a lovely practical payoff:

> We do **not** need a complicated joint sampler. We can draw the waiting time $\tau$ and the
> reaction index $j$ **separately**, using two independent random numbers.

And the probability that the next reaction is $R_j$ is simply

$$\boxed{\;\Pr(j \mid \mathbf{x}) \;=\; \frac{a_j(\mathbf{x})}{a_0(\mathbf{x})}\;}$$

— reactions fire in proportion to how "eager" they are. A reaction with twice the propensity is
twice as likely to be the next one.

## 8. How $j$ is actually obtained — and what $r_2$ is

It remains to draw a reaction index $j$ with probabilities $a_j/a_0$. Picture the interval
$[0,\,a_0]$ sliced into $M$ consecutive pieces, where piece $j$ has **length** $a_j$:

$$\underbrace{[\,0,\, a_1\,]}_{R_1}\;\Big|\;\underbrace{[\,a_1,\, a_1{+}a_2\,]}_{R_2}\;\Big|\;\cdots\;\Big|\;\underbrace{[\,\textstyle\sum_{k<M}\!a_k,\, a_0\,]}_{R_M}.$$

Pick a uniform random point on $[0, a_0]$ and ask which piece it falls into. Piece $j$ is hit with
probability $\dfrac{\text{its length}}{\text{total length}} = \dfrac{a_j}{a_0}$ — exactly what we want.
This is the classic "**roulette wheel**" (or cumulative-sum) method.

To generate that uniform point, draw $r_2 \sim U(0,1)$ and scale it up: the point is $r_2 \cdot a_0$.
Then scan the cumulative propensities $c_j = a_1 + a_2 + \cdots + a_j$ and pick the **first** $j$
with $c_j > r_2\,a_0$:

$$\boxed{\;\text{choose } j \text{ such that } \sum_{k=1}^{j-1} a_k \;\le\; r_2\,a_0 \;<\; \sum_{k=1}^{j} a_k\;}$$

> **Worked example.** Suppose $a = (1,\,2,\,2)$, so $a_0 = 5$. The wheel is
> $[\,0,1\,]\!=\!R_1$, $\,[\,1,3\,]\!=\!R_2$, $\,[\,3,5\,]\!=\!R_3$.
> * $r_2 = 0.1 \Rightarrow r_2 a_0 = 0.5 \in [0,1] \Rightarrow R_1$.
> * $r_2 = 0.5 \Rightarrow 2.5 \in [1,3] \Rightarrow R_2$.
> * $r_2 = 0.9 \Rightarrow 4.5 \in [3,5] \Rightarrow R_3$.
>
> Over many draws, $R_1$ is picked $1/5 = 20\%$ of the time, $R_2$ and $R_3$ each $2/5 = 40\%$.

> **So what is $r_2$?** A second, *independent* uniform random number on $(0,1)$ — the sole source
> of randomness for **which** reaction fires. Its independence from $r_1$ is exactly the
> independence of $\tau$ and $j$ we proved above.

## 9. Numerical sanity check (1): is $\tau$ really exponential?

Before trusting the theory, let's test it. Fix $a_0 = 5$ and draw many waiting times exactly as
the algorithm does, $\tau = \tfrac{1}{a_0}\ln(1/r_1)$. The histogram should overlay the
$\mathrm{Exp}(5)$ density, with mean $\approx 1/5 = 0.2$ and variance $\approx 1/25 = 0.04$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
a0 = 5.0
n = 200_000

r1 = rng.random(n)
tau = (1.0 / a0) * np.log(1.0 / r1)

grid = np.linspace(0, tau.max(), 400)
plt.figure(figsize=(7, 4))
plt.hist(tau, bins=120, density=True, alpha=0.55, label=r'sampled $\tau$')
plt.plot(grid, a0 * np.exp(-a0 * grid), 'r-', lw=2.2, label=rf'Exp({a0:g}) pdf')
plt.xlabel(r'waiting time $\tau$'); plt.ylabel('density')
plt.title('Waiting time matches Exp($a_0$)')
plt.legend(); plt.tight_layout(); plt.show()

print(f'mean tau = {tau.mean():.4f}   (theory {1/a0:.4f})')
print(f'var  tau = {tau.var():.4f}   (theory {1/a0**2:.4f})')

## 10. Numerical sanity check (2): is each reaction picked with probability $a_j/a_0$?

Using the same wheel logic, draw $j$ many times for propensities $a = (1, 2, 2)$ and compare the
empirical frequencies with the theory $(0.2,\, 0.4,\, 0.4)$.

In [ ]:
a = np.array([1.0, 2.0, 2.0])
a0 = a.sum()
cum = np.cumsum(a)

r2 = rng.random(n)
chosen = np.searchsorted(cum, r2 * a0, side='right')   # which piece of the wheel

emp = np.bincount(chosen, minlength=3) / n
theory = a / a0
print('reaction :  empirical  |  theory a_j/a_0')
for j in range(3):
    print(f'   R{j+1}    :   {emp[j]:.4f}    |    {theory[j]:.4f}')

plt.figure(figsize=(5.5, 4))
plt.bar(['R1', 'R2', 'R3'], emp, alpha=0.55, label='sampled')
plt.plot(['R1', 'R2', 'R3'], theory, 'ro', ms=11, label='theory')
plt.ylabel('probability'); plt.title(r'Reaction selection matches $a_j/a_0$')
plt.legend(); plt.tight_layout(); plt.show()

## 11. The complete algorithm (the *Direct Method*)

Every ingredient is now derived and justified. One step of the simulation is:

1. **State & propensities.** Given the current state $\mathbf{X}$, compute each $a_j(\mathbf{X})$
   and $a_0 = \sum_j a_j$. *(If $a_0 = 0$, no reaction can ever fire — stop.)*
2. **Draw two independent uniforms.** $r_1, r_2 \sim U(0,1)$.
3. **When** — sample the waiting time: $\displaystyle\quad \tau = \frac{1}{a_0}\ln\!\left(\frac{1}{r_1}\right).$
   *(Justified: $\tau \sim \mathrm{Exp}(a_0)$, §5–6.)*
4. **Which** — pick the reaction: $\quad j = $ first index with $\sum_{k=1}^{j} a_k > r_2\,a_0$.
   *(Justified: $\Pr(j) = a_j/a_0$, §7–8.)*
5. **Update.** Advance the clock $t \leftarrow t + \tau$, and apply the stoichiometry
   $\mathbf{X} \leftarrow \mathbf{X} + \boldsymbol{\nu}_j$.
6. **Record** $(t, \mathbf{X})$ and go back to step 1 until $t$ exceeds the desired end time.

That is the entire algorithm — two random numbers per reaction event, each with a clear meaning.

## 12. A worked simulation: stochastic gene expression

We model a gene being transcribed into mRNA, which is translated into protein, with both
molecules degrading:

| Reaction | Stoichiometry $\boldsymbol{\nu}$ | Propensity |
|---|---|---|
| $\varnothing \to$ mRNA | $(+1,\,0)$ | $k_m$ |
| mRNA $\to$ mRNA + P | $(0,\,+1)$ | $k_p \cdot$ mRNA |
| mRNA $\to \varnothing$ | $(-1,\,0)$ | $\gamma_m \cdot$ mRNA |
| P $\to \varnothing$ | $(0,\,-1)$ | $\gamma_p \cdot$ P |

We implement the Direct Method exactly as above and run one trajectory. (Steady-state averages for
this model are $\langle\text{mRNA}\rangle = k_m/\gamma_m$ and
$\langle\text{P}\rangle = k_p k_m/(\gamma_m\gamma_p)$ — a good check that the simulation is correct.)

In [ ]:
def gillespie_ssa(prop_func, stoich, x0, t_end, rng):
    # stoich[j] is the state-change vector nu_j; prop_func(x) returns the a_j(x).
    x = np.array(x0, dtype=float); t = 0.0
    times = [t]; states = [x.copy()]
    while True:
        a = np.asarray(prop_func(x), dtype=float)
        a0 = a.sum()
        if a0 <= 0.0:
            break
        r1, r2 = rng.random(), rng.random()
        tau = (1.0 / a0) * np.log(1.0 / r1)                 # step 3
        j = int(np.searchsorted(np.cumsum(a), r2 * a0, side='right'))  # step 4
        t += tau
        if t > t_end:
            break
        x = x + stoich[j]                                    # step 5
        times.append(t); states.append(x.copy())
    return np.array(times), np.asarray(states)

# parameters and reaction definitions
k_m, k_p, gm, gp = 2.0, 10.0, 0.5, 0.2
def gene_prop(x):
    m, p = x
    return np.array([k_m, k_p * m, gm * m, gp * p])
gene_stoich = np.array([[ 1, 0], [0, 1], [-1, 0], [0, -1]], dtype=float)

t_tr, X_tr = gillespie_ssa(gene_prop, gene_stoich, [0, 0], 60.0, np.random.default_rng(42))

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].step(t_tr, X_tr[:, 0], where='post', color='C0'); ax[0].set_ylabel('mRNA')
ax[0].set_title('Gillespie SSA: stochastic gene expression (one trajectory)')
ax[1].step(t_tr, X_tr[:, 1], where='post', color='C3'); ax[1].set_ylabel('protein'); ax[1].set_xlabel('time')
plt.tight_layout(); plt.show()

print(f'mean mRNA = {X_tr[:,0].mean():.2f}   (theory {k_m/gm:.2f})')
print(f'mean prot = {X_tr[:,1].mean():.2f}   (theory {k_p*k_m/(gm*gp):.2f})')

## 13. Summary — the whole derivation on one page

**The chain of reasoning.** From the single definition of propensity $a_j\,dt$, we obtained:

| Step | Result | Reason |
|---|---|---|
| Survival function | $P_0(\tau) = e^{-a_0\tau}$ | $(1 - a_0\tau/n)^n \to e^{-a_0\tau}$ as $n\to\infty$ |
| Waiting-time density | $f(\tau) = a_0 e^{-a_0\tau}$, so $\tau \sim \mathrm{Exp}(a_0)$ | $P_0(\tau)\cdot a_0$ |
| How to draw $\tau$ | $\tau = \tfrac{1}{a_0}\ln(1/r_1)$ | inverse-CDF of the exponential |
| Reaction identity | $\Pr(j) = a_j/a_0$ | factorising $p(\tau,j) = a_j e^{-a_0\tau}$ |
| How to draw $j$ | first $j$ with $\sum_{k\le j} a_k > r_2\,a_0$ | roulette wheel over the propensities |

**The two random numbers, restated.**

* $r_1 \sim U(0,1)$ becomes the **waiting time** $\tau = \frac{1}{a_0}\ln(1/r_1)$ — it answers **when**.
* $r_2 \sim U(0,1)$ becomes the **reaction index** $j$ via the propensity wheel — it answers **which**.

Their independence mirrors the factorisation $p(\tau, j) = f(\tau)\cdot\Pr(j)$: the *timing* of the
next event and *which* event it is have nothing to do with each other. That is the entire content of
Gillespie's algorithm — every formula used in practice follows from these few lines.